# 이화여대 Day0/Day12 — GATK 4.7.0.0 HaplotypeCaller (Haploid)

**입력:** Drive `Day0_Day12_Variant_Calling/01.RawData`의 FASTQ 6쌍  
**참조:** `outputs/reference/day0_common_reference.fasta` (Day0.1, 4 contigs, 2,822,641 bp, SHA256 고정)  
**실행:** FASTQ → BWA-MEM → coordinate-sort → GATK MarkDuplicates → HaplotypeCaller (ploidy=1, GVCF) → CombineGVCFs → GenotypeGVCFs → 정규화 → LoFreq 대조  
**출력:** `outputs/gatk_haploid_v1` (샘플별 GVCF, BAM, 중복 지표, QC 로그, 공동 VCF, 후보 CSV).

이 노트북은 원본 파일을 삭제·수정하지 않습니다. 재실행 시 PASS 결과를 재사용합니다. **GATK 결과는 주로 우세한 haploid 대립유전자 검출용이며, 저빈도 변이는 기존 LoFreq v9.1 검증을 우선합니다.**

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Day0_Day12_Variant_Calling')
SAMPLES = ['Day0.1','Day0.2','Day0.3','Day12.1','Day12.2','Day12.3']
assert ROOT.is_dir(), f'Google Drive project missing: {ROOT}'
for s in SAMPLES:
    for mate in ('1','2'):
        f = ROOT / '01.RawData' / s / f'{s}_L1_{mate}.fq.gz'
        assert f.is_file() and f.stat().st_size > 10000, f'Missing FASTQ: {f}'
print('Raw FASTQ verified: 12/12')
print('Reference:', ROOT/'outputs/reference/day0_common_reference.fasta')

In [ ]:
import subprocess, sys, urllib.request, hashlib
subprocess.run([sys.executable,'-m','pip','-q','install','pysam>=0.22,<1','pandas>=2.0,<3'],check=True)
# Reproducible source maintained in GitHub; raw reads remain only in Drive.
SCRIPT_URL = 'https://raw.githubusercontent.com/kimtk94/Codex/d4dc5fc0ecd7de488569d62e213e40b3cdfbb14b/research/ewha_day0_day12/gatk_haploid_pipeline.py'
SCRIPT = '/content/gatk_haploid_pipeline.py'
urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
print('Pipeline SHA256:',hashlib.sha256(open(SCRIPT,'rb').read()).hexdigest())
subprocess.run([sys.executable,SCRIPT,'--help'],check=True)

## 1. Day0.1 파일럿

먼저 대표 샘플 하나에 대해 FASTQ 압축 무결성, BWA 정렬, duplicate marking, GATK GVCF 생성 및 index 검사를 진행합니다. 성공하면 2번 셀에서 나머지 5개를 처리합니다.

In [ ]:
subprocess.run([sys.executable,SCRIPT,'--pilot','--keep-bam'],check=True)
print((ROOT/'outputs/gatk_haploid_v1/samples/Day0.1/status.json').read_text())

## 2. 전체 6샘플 + 공동 유전형 판정

각 샘플의 GVCF를 확인하고 공동 GenotypeGVCFs를 수행합니다. 이전 PASS 결과는 재사용하며 최종 VCF와 LoFreq 비교표를 Google Drive에 저장합니다.

In [ ]:
subprocess.run([sys.executable,SCRIPT,'--keep-bam'],check=True)

In [ ]:
import json, pandas as pd
output = ROOT/'outputs/gatk_haploid_v1/cohort'
summary = json.loads((output/'summary.json').read_text())
print(json.dumps(summary,ensure_ascii=False,indent=2))
df = pd.read_csv(output/'joint_variant_audit.csv')
display(df['classification'].value_counts().to_frame('count'))
display(df[df['classification'].isin(['REPLICATED_AND_LOFREQ','REPLICATED_GATK_ONLY_REVIEW'])].head(30))
print('Joint VCF:',output/'joint.normalized.vcf.gz')
print('Full candidate CSV:',output/'joint_variant_audit.csv')

## 해석 기준

- 6개 biological specimen의 genotype과 read depth를 개별 확인합니다.
- `REPLICATED_AND_LOFREQ`: Day0 3개 샘플에서 기준 대립유전자 검출 가능(DP≥10, GQ≥30), Day12 ≥2개에서 동일 ALT (DP≥10, GQ≥30), LoFreq 해당 allele 일치.
- `REPLICATED_GATK_ONLY_REVIEW`: 동일하지만 LoFreq 불일치, 독립 BAM pileup 필요.
- `SINGLE_DAY12_REVIEW`: Day12 단일 샘플에서 발견되어 반복 재현성 부족.
- GATK의 haploid GT는 혼합집단의 저빈도 allele frequency를 정량하지 않습니다.
- Day0.1 참조에 없는 accessory DNA, plasmid, 구조변이는 별도 검증이 필요합니다.
- BQSR/VQSR는 적합한 known-sites truth set이 확인되지 않아 시행하지 않습니다.